In [1]:
from collections import OrderedDict
from typing import List, Tuple, Optional, Union
import copy, os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets.utils.logging import disable_progress_bar
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter


import flwr
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import Metrics, Context
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr.server.client_proxy import ClientProxy
from flwr.common import (
    FitRes,
    Parameters,
    Scalar,
)


import argparse
import pandas as pd
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from imblearn.over_sampling import RandomOverSampler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# DEVICE = torch.device("cpu")
print(f"Training on {DEVICE}")
print(f"Flower {flwr.__version__} / PyTorch {torch.__version__}")
disable_progress_bar()

/home/ehsan/miniconda3/envs/flower/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-01-23 14:11:56.980887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737670316.993436   16532 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737670316.997274   16532 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-23 14:11:57.009618: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-cr

Training on cuda
Flower 1.14.0 / PyTorch 2.5.1


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "fed_avg_flower"
meta_args.model = "mlp"

# meta_args.model = "cnn"SO FAR GOOD WITHOUT NORMALIZATION 
# meta_args.round = 20
# meta_args.epoch_iterations = 20
# meta_args.local_lr = 0.001
# meta_args.batch_size = 100
# meta_args.decay_weight = 1.0
# meta_args.data_type = ""
meta_args.round = 80 # 50
meta_args.epoch_iterations = 20
meta_args.local_lr = 0.001
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.remove_labels = [17, 21, 25, 29]
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    }

print(meta_args)

Namespace(gpu=0, seed=1, repeat=1, device=device(type='cuda', index=0), log_path='fed_avg_flower', model='mlp', round=80, epoch_iterations=20, local_lr=0.001, batch_size=150, decay_weight=1.0, data_type='', remove_labels=[17, 21, 25, 29], features=['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max', 'sender_nic_send_bytes', 'sender_nic_receive_bytes', 'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes'], filenames={'wisconsin_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv', 'wisconsin_hdd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv', 'wisconsin_hdd_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv', 'wisconsin_ssd_delay_10ms_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv'})


In [3]:
class MLPClassifier_torch(nn.Module):
    def __init__(self, input_size, output_size=2, hidden_layer_sizes=(100,),
                 learning_rate=0.001, max_iter=200, tol=1e-4, random_state=None):
        super(MLPClassifier_torch, self).__init__()

        if random_state is not None:
            torch.manual_seed(random_state)

        # Create the network architecture
        layers = []
        prev_size = input_size
        for size in hidden_layer_sizes:
            layers.append(nn.Linear(prev_size, size))
            layers.append(nn.BatchNorm1d(size))
            layers.append(nn.ReLU())
            prev_size = size
        layers.append(nn.Linear(prev_size, output_size))
        # layers.append(nn.Softmax(dim=1))  # Softmax for multi-class classification

        self.net = nn.Sequential(*layers)
        # self.learning_rate = learning_rate
        # self.max_iter = max_iter
        # self.tol = tol
        self.optimizer = None
        # self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss
        self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss

    def forward(self, x):
        return self.net(x)

In [4]:
def process_and_prepare_loaders(args, remove_labels=None, features=None, filenames=None):
    if remove_labels is None:
        remove_labels = [17, 21, 25, 29]
    if features is None:
        features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
                    'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
    if filenames is None:
        filenames = {
            "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
            # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

            "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
            # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",
        }

    clients_data_loaders = {}
    client_test_loaders = {}
    combined_X_test, combined_y_test = [], []
    test_data_dict = {}

    for client_name, file_path in filenames.items():
        # Step 1: Load the dataset and Label encoding and scaling
        df = pd.read_csv(file_path)
        
        # Step 2: Remove specified labels
        for lbl in remove_labels:
            df = df.drop(df[df.label_value == lbl].index)
        
        # Normalize for transfer learning 
        # df = normalize_df(df)

        X = df.drop(columns="label_value")[features]
        y = df.label_value

        encoder = LabelEncoder()
        scaler = StandardScaler()
        
        y = encoder.fit_transform(y)
        # X = scaler.fit_transform(X)

        # Step 3: Split into train and test sets
        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X,y)
        
       
        # X_train = scaler.fit_transform(X_train)
        # X_test = scaler.transform(X_test)

        # Step 4: Apply oversampling to training data
        X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)

        X_train = X_train.to_numpy() if not isinstance(X_train, np.ndarray) else X_train
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_train = y_train.to_numpy() if not isinstance(y_train, np.ndarray) else y_train
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test


        # Step 5: Create train DataLoader
        train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                    torch.tensor(y_train, dtype=torch.long))

        ldr_train = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
        # data_loader_list.append(ldr_train)
        clients_data_loaders[client_name] = ldr_train

        # Combine test data for unified test dataset
        combined_X_test.append(X_test)
        combined_y_test.append(y_test)

        # Create individual test DataLoader
        test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                      torch.tensor(y_test, dtype=torch.long))
        
        client_test_loaders[client_name] = DataLoader(test_dataset, batch_size=args.batch_size)

        
    # Combine all test data
    combined_X_test = np.vstack(combined_X_test)
    combined_y_test = np.hstack(combined_y_test)
    total_classes = len(np.unique(combined_y_test))
     # Create combined test DataLoader
    combined_test_dataset = TensorDataset(torch.tensor(combined_X_test, dtype=torch.float32),
                                           torch.tensor(combined_y_test, dtype=torch.long))
    global_test_loader = DataLoader(combined_test_dataset, batch_size=args.batch_size, shuffle=False)


    args.input_size = len(features)
    args.output_size = total_classes

    return clients_data_loaders, client_test_loaders, global_test_loader, total_classes, args 


In [5]:
def set_log_path(args):
    import datetime
    path =  './log/' + args.log_path+ '/'
    if not os.path.exists(path):
        os.makedirs(path)
    path_log = os.path.join(path)
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    return path_log + '_' + str(timestamp)

def summarize_dataloader(dataloader):
    print("=== DataLoader Summary ===")
    # Dataset length
    dataset_size = len(dataloader.dataset)
    print(f"Total samples: {dataset_size}")
    
    # Batch size
    batch_size = dataloader.batch_size
    print(f"Batch size: {batch_size}")
    
    # Number of batches
    num_batches = len(dataloader)
    print(f"Number of batches: {num_batches}")
    
    # Inspect a single batch
    for i, batch in enumerate(dataloader):
        print(f"Inspecting Batch {i+1}:")
        if isinstance(batch, dict):
            for key, value in batch.items():
                if isinstance(value, (list, tuple)):
                    print(f"  {key}: List/Tuple of length {len(value)}")
                else:
                    print(f"  {key}: Shape {value.shape}, Type {value.dtype}")
        elif isinstance(batch, (list, tuple)):
            for idx, value in enumerate(batch):
                if isinstance(value, torch.Tensor):
                    print(f"  Element {idx}: Shape {value.shape}, Type {value.dtype}")
                else:
                    print(f"  Element {idx}: Type {type(value)}")
        else:
            print("  Batch is not a dict, list, or tuple. Unexpected format.")
        # Only inspect the first batch
        break

    print("===========================")

In [6]:
def load_datasets(partition_id: int):
    client_name = list(args.filenames.keys())[int(partition_id)]
    
    trainloader = clients_data_loaders[client_name]
    testloader = client_test_loaders[client_name]
    valloader = testloader

    return trainloader, valloader, testloader

# load_datasets(2)

In [7]:
args = copy.deepcopy(meta_args)
clients_data_loaders, client_test_loaders, global_test_loader, total_classes, args = process_and_prepare_loaders(args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)

print(clients_data_loaders, "\n")
print(client_test_loaders, "\n")
summarize_dataloader(client_test_loaders["wisconsin_ssd_merged"])

{'wisconsin_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x748758966210>, 'wisconsin_hdd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7487599cbce0>, 'wisconsin_hdd_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x74875ada5100>, 'wisconsin_ssd_delay_10ms_merged': <torch.utils.data.dataloader.DataLoader object at 0x74875897ffb0>} 

{'wisconsin_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x74875946b410>, 'wisconsin_hdd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7487589aa1b0>, 'wisconsin_hdd_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x748758bb9f10>, 'wisconsin_ssd_delay_10ms_merged': <torch.utils.data.dataloader.DataLoader object at 0x748758b74170>} 

=== DataLoader Summary ===
Total samples: 1408
Batch size: 150
Number of batches: 10
Inspecting Batch 1:
  Element 0: Shape torch.Size([150, 12]), Type torch.float32
  Element 1: Shape torch.Size([150]), Type torch.int64


In [8]:
def train(net, ldr_train, epochs: int, device, verbose=False, local_lr=0.001):
    loss_func = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=local_lr)
    epochs_losses = []
    net.train()
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for _, (batch_X, labels) in enumerate(ldr_train):
            batch_X, labels = batch_X.to(device), labels.to(device)
            net.zero_grad()
            # optimizer.zero_grad()
            log_probs = net.forward(batch_X)
            loss = loss_func(log_probs, labels)
            loss.backward()
            optimizer.step()
            # Metrics
            epochs_losses.append(loss.item())
            epoch_loss += loss.item()
            total += labels.size(0)
            correct += (torch.max(log_probs.data, 1)[1] == labels).sum().item()
        if verbose:
            epoch_acc = correct / total
            epoch_loss /= len(ldr_train.dataset)
            print(f"Epoch {epoch+1}: train loss {epoch_loss}, accuracy {epoch_acc}")
    w_new = copy.deepcopy(net.state_dict())
    return w_new, sum(epochs_losses) / len(epochs_losses)



def test(net, ldr_test, device):
    net = copy.deepcopy(net).to(device)
    loss_func = nn.CrossEntropyLoss()
    net.eval()
    correct, total, test_loss = 0, 0, 0.0
    
    all_preds, all_targets = [], []

    with torch.no_grad():
        for index, (data, target) in enumerate(ldr_test):
             data, target = data.to(device), target.to(device)
             log_probs = net.forward(data)
             test_loss += loss_func(log_probs, target).item()
             _, predicted = torch.max(log_probs, -1) # TODO CHECK FOR GET -1 pr 1 is correct
             
             total += target.size(0)
             correct += predicted.eq(target).sum()
             all_preds.extend(predicted.cpu().numpy())
             all_targets.extend(target.cpu().numpy())
    test_loss /= len(ldr_test.dataset)
    accuracy = 100.00 * correct.item() / total
    f1 = f1_score(all_targets, all_preds, average='weighted')
    return test_loss, accuracy, f1



In [10]:
trainloader, valloader, testloader = load_datasets(partition_id=0)
# net = MLPClassifier_torch().to(DEVICE)
net = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(200,)).to(args.device)

for epoch in range(3):
    _, train_loss = train(net, trainloader, 2,device=args.device ,local_lr=args.local_lr)
    print("train loss", train_loss)
    loss, accuracy, f1_csore = test(net, valloader, args.device)
    print(f"Epoch {epoch+1}: test loss {loss}, accuracy {accuracy}, f1_score {f1_csore}")

train loss 1.0859006649377394
Epoch 1: test loss 0.007208815902810205, accuracy 56.03693181818182, f1_score 0.48621642338041854
train loss 0.7071695583207267
Epoch 2: test loss 0.005841130052100529, accuracy 65.05681818181819, f1_score 0.623164341308393
train loss 0.5654804837338778
Epoch 3: test loss 0.004378336058421569, accuracy 79.1903409090909, f1_score 0.7746712483753607


In [17]:
# def set_parameters(net, parameters: List[np.ndarray]):
#     params_dict = zip(net.state_dict().keys(), parameters)
#     state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
#     net.load_state_dict(state_dict, strict=True)


# def get_parameters(net) -> List[np.ndarray]:
#     return [val.cpu().numpy() for _, val in net.state_dict().items()]

In [46]:
class FlowerClient(NumPyClient):
    def __init__(self, net, trainloader, valloader, partition_id):
        self.p_id = partition_id
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader
        

    # def get_parameters(self, config):
    #     return get_parameters(self.net)
    def get_parameters(self, config) -> List[np.ndarray]:
        # Return model parameters as a list of NumPy ndarrays
        # Exclude parameters of BN layers when using FedBN
        return [
            val.cpu().numpy()
            for name, val in self.net.state_dict().items()
            if "bn" not in name
        ]

    def set_parameters(self, parameters: List[np.ndarray]) -> None:
        # Set model parameters from a list of NumPy ndarrays
        keys = [k for k in self.net.state_dict().keys() if "bn" not in k]
        params_dict = zip(keys, parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.net.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        print(f"Client {self.p_id} starting fit")
        # self.set_parameters(self.net, parameters)
        self.set_parameters(parameters)
        _, train_loss = train(self.net, self.trainloader, epochs=args.epoch_iterations, device=args.device, verbose=False ,local_lr=args.local_lr)
        loss, accuracy, f1_score = test(self.net, self.trainloader, device=args.device) 
        # return self.get_parameters(self.net), len(self.trainloader), {"loss": loss, "accuracy":  float(accuracy), "f1_score": f1_score, "train_local_loss": train_loss}
        return self.get_parameters(config), len(self.trainloader), {"loss": loss, "accuracy":  float(accuracy), "f1_score": f1_score, "train_local_loss": train_loss}


    def evaluate(self, parameters, config):
        # self.set_parameters(self.net, parameters)
        self.set_parameters(parameters)
        loss, accuracy, f1_score = test(self.net, self.valloader, args.device)
        return float(loss), len(self.valloader), {"accuracy": float(accuracy), "loss": float(loss), "f1_score": float(f1_score)}
    
def client_fn(context: Context) -> Client:
    """Create a Flower client representing a single organization."""

    # Load model
    net = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(200,)).to(args.device)

    # Load data (CIFAR-10)
    # Note: each client gets a different trainloader/valloader, so each client
    # will train and evaluate on their own unique data partition
    # Read the node_config to fetch data partition associated to this node
    partition_id = context.node_config["partition-id"]
    trainloader, valloader, _ = load_datasets(partition_id=partition_id)

    # Create a single Flower client representing a single organization
    # FlowerClient is a subclass of NumPyClient, so we need to call .to_client()
    # to convert it to a subclass of `flwr.client.Client`
    return FlowerClient(net, trainloader, valloader, partition_id).to_client()


# Create the ClientApp
client = ClientApp(client_fn=client_fn)

In [47]:
def get_evaluate_fn(testloader):
    """Return a function that can be called to do global evaluation."""

    def evaluate_fn(server_round: int, parameters, config):
        """Evaluate global model on the whole test set."""

        model = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(200,)).to(args.device)
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model.to(device)


        # Set model parameters from a list of NumPy ndarrays
        keys = [k for k in model.state_dict().keys() if "bn" not in k]
        params_dict = zip(keys, parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        model.load_state_dict(state_dict, strict=False)

        
        # # set parameters to the model
        # params_dict = zip(model.state_dict().keys(), parameters)
        # state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
        # model.load_state_dict(state_dict, strict=True)

        # call test (evaluate model as in centralised setting)
        loss, accuracy, f1_score = test(model, testloader, args.device)
        # print(f"Round {server_round} - Evaluation: loss {loss}, accuracy {accuracy}, f1_score {f1_score}")
        return loss, {"accuracy": accuracy, "loss": loss, "f1_score": f1_score}

    return evaluate_fn

# Define metric aggregation function
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # print("\nPrinting metrics in weighted_average function \n {} \n".format(metrics))

    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metric (weighted average)
    return {"accuracy": sum(accuracies) / sum(examples)}

def fit_metrics_aggregation_fn(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # print("\nPrinting metrics in fit_metrics_aggregation_fn function \n {} \n".format(metrics))

    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    losses = [num_examples * m["loss"] for num_examples, m in metrics]
    train_loss = [m["train_local_loss"] for _, m in metrics]
    f1_scores = [num_examples * m["f1_score"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, m in metrics]
    # return {"accuracy": sum(accuracies) / sum(examples), "loss": sum(losses) / sum(examples), "f1_score": sum(f1_scores) / sum(examples), "train_loss": sum(train_loss) / len(train_loss)}
    return {"accuracy": sum(accuracies) / sum(examples), "loss": sum(train_loss) / len(train_loss), "f1_score": sum(f1_scores) / sum(examples), "train_loss": sum(train_loss) / len(train_loss)}

In [48]:
class FedAvgCustom(FedAvg):
    def __init__(self, meta_args, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        # Run simulation
        print("{:<50}".format("-" * 15 + " log path " + "-" * 50)[0:60])
        log_path = set_log_path(meta_args)
        print(log_path)
        self.writer = SummaryWriter(log_path)
    
    def aggregate_fit(self, server_round: int, results: list[tuple[ClientProxy, FitRes]], failures: list[Union[tuple[ClientProxy, FitRes], BaseException]],):
        parameters_aggregated, metrics_aggregated = super().aggregate_fit(server_round, results, failures)
        # print(f"Round {server_round} - Aggregated fit: {metrics_aggregated}")
        self.writer.add_scalar("train_loss", metrics_aggregated["train_loss"], server_round)
        return parameters_aggregated, metrics_aggregated

    def evaluate(self, server_round: int, parameters: Parameters):
        loss, metrics = super().evaluate(server_round, parameters)
        print(f"Round {server_round} - Evaluation: {metrics}")
        self.writer.add_scalar("test_accuracy", metrics["accuracy"], server_round)
        self.writer.add_scalar("test_loss", loss, server_round)
        self.writer.add_scalar("test_f1_score", metrics["f1_score"], server_round)

In [49]:
# Create FedAvg strategy
# strategy = FedAvg(
#     fraction_fit=1.0,  # Sample 100% of available clients for training
#     fraction_evaluate=0.5,  # Sample 50% of available clients for evaluation
#     min_fit_clients=4,  # Never sample less than 10 clients for training
#     min_evaluate_clients=4,  # Never sample less than 5 clients for evaluation
#     min_available_clients=4,
#     evaluate_metrics_aggregation_fn=weighted_average, # callback defined earlier
#     fit_metrics_aggregation_fn=fit_metrics_aggregation_fn,  # callback defined earlier
#     evaluate_fn=get_evaluate_fn(
#         global_test_loader, 
#     ),  # Wait until all 3 clients are available
# )

strategy = FedAvgCustom(
    meta_args=args,
    fraction_fit=1.0,  # Sample 100% of available clients for training
    fraction_evaluate=0.5,  # Sample 50% of available clients for evaluation
    min_fit_clients=4,  # Never sample less than 10 clients for training
    min_evaluate_clients=4,  # Never sample less than 5 clients for evaluation
    min_available_clients=4,
    evaluate_metrics_aggregation_fn=weighted_average, # callback defined earlier
    fit_metrics_aggregation_fn=fit_metrics_aggregation_fn,  # callback defined earlier
    evaluate_fn=get_evaluate_fn(
        global_test_loader, 
    ),  # Wait until all 3 clients are available
)

def server_fn(context: Context) -> ServerAppComponents:
    """Construct components that set the ServerApp behaviour.

    You can use the settings in `context.run_config` to parameterize the
    construction of all elements (e.g the strategy or the number of rounds)
    wrapped in the returned ServerAppComponents object.
    """

    # Configure the server for 5 rounds of training
    # config = ServerConfig(num_rounds=args.round)
    config = ServerConfig(num_rounds=2)
    
    return ServerAppComponents(strategy=strategy, config=config)

# Create the ServerApp
server = ServerApp(server_fn=server_fn)

run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=4,
    backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 1.0}}
)

INFO :      Starting Flower ServerApp, config: num_rounds=2, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client


--------------- log path -----------------------------------
./log/fed_avg_flower/_2025-01-21_18-48-10


(pid=535117) 2025-01-21 18:48:13.915192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=535117) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=535117) E0000 00:00:1737514093.928579  535117 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=535117) E0000 00:00:1737514093.932512  535117 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=535117) 2025-01-21 18:48:13.945163: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(pid=535117) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFl

Round 0 - Evaluation: {'accuracy': 12.04669260700389, 'loss': 301262.0887159533, 'f1_score': 0.07166077905180442}
(ClientAppActor pid=535118) Client 0 starting fit


INFO :      aggregate_fit: received 4 results and 0 failures
INFO :      configure_evaluate: strategy sampled 4 clients (out of 4)


...

INFO :      aggregate_fit: received 4 results and 0 failures
INFO :      configure_evaluate: strategy sampled 4 clients (out of 4)


Round 2 - Evaluation: {'accuracy': 37.68093385214008, 'loss': 0.04410518653661824, 'f1_score': 0.3665371972559354}


INFO :      aggregate_evaluate: received 4 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 2 round(s) in 11.12s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.022355829056538472
INFO :      		round 2: 0.046328147214160675
INFO :      	History (metrics, distributed, fit):
INFO :      	{'accuracy': [(1, 90.75585372486167), (2, 89.14567250023991)],
INFO :      	 'f1_score': [(1, 0.8963599873274879), (2, 0.8751214081336045)],
INFO :      	 'loss': [(1, 0.37344471466942325), (2, 0.2099909449284376)],
INFO :      	 'train_loss': [(1, 0.37344471466942325), (2, 0.2099909449284376)]}
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 55.92283957244445), (2, 37.60113874089393)]}
INFO :      


(ClientAppActor pid=535119) Client 2 starting fit [repeated 3x across cluster]


(pid=535118) 2025-01-21 18:48:14.015785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered [repeated 4x across cluster]
(pid=535118) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 4x across cluster]
(pid=535118) E0000 00:00:1737514094.029717  535118 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 4x across cluster]
(pid=535119) E0000 00:00:1737514093.983294  535119 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 4x across cluster]
(pid=535119) 2025-01-21 18:48:13.996045: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-cri

In [ ]:
import flwr

